In [1]:
import pandas as pd

df_1 = pd.read_csv('file.csv')

df_2 = pd.read_csv('file2.csv')

catalog_names = df_1['catalog_items']

df_3 = []

for i, row in  df_2.iterrows():
    for x in row:
        if x not in catalog_names:
            df_3.append(row)



new_df = pd.DataFrame(df_3)
new_df.to_csv("output.csv")
    

FileNotFoundError: [Errno 2] No such file or directory: 'file.csv'

In [ ]:
import os
for root,dirs,files in os.walk('\mnt\f\LimeLight-Video\limelight_video\media\videos'):
    print(root)
    print(dirs)
    print(files)
    for file in files:
        print(file)
        print(dirs)

In [ ]:
import django_postgrespool2

In [11]:
import os
import json
import cv2
import subprocess
import logging
from statistics import median
from bertopic import BERTopic

import torch
import whisper
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer
from concurrent.futures import ThreadPoolExecutor

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import IncrementalPCA

from bertopic.vectorizers import OnlineCountVectorizer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

from PIL import Image
from scipy.special import expit
from skimage.metrics import structural_similarity as ssim
from transformers import pipeline
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer, AutoModelForSequenceClassification

from django.conf import settings
#from transcoder.models import VideoEngineVideopost


!pip install opencv-python
!pip install scikit-image 
!pip install tensorrt
!pip install openai-whisper
!pip install setuptools-rust



from sentence_transformers import SentenceTransformer



class Ronin:
    
    def __scene_classification_meta__(self):
        self.model = VisionEncoderDecoderModel.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
        self.feature_extractor = ViTImageProcessor.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
        self.tokenizer = AutoTokenizer.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.max_length = 16
        self.num_beams = 4
        self.gen_kwargs = {"max_length": self.max_length, "num_beams": self.num_beams}

    def load_sub_models(self):
        self.embedding_model = SentenceTransformer("all-distilroberta-v1")
        self.umap_model = IncrementalPCA(n_components=256)
        self.cluster_model = MiniBatchKMeans(n_clusters=2048, random_state=0)
        self.vectorizer_model = OnlineCountVectorizer(stop_words="english", decay=.01)
        self.representation_model = {
            "KeyBERT": KeyBERTInspired(),
            "MMR": MaximalMarginalRelevance(diversity=0.4)

        }
        logging.info("-> Default sub-models loaded. ")


    def __topic_modeller_meta__(self):
        self.topic_modellers_available =  ['2Mil_C4', "4Mil_C8_Heavy", "5Mil_C10_Lite"]
        self.topic_model_type = "5Mil_C10_Lite"
        self.load_sub_models()
        nltk.download('punkt')
        nltk.download('wordnet')
        nltk.download('stopwords')
        
    def __meta__(self) -> None:
        self.VERSION = 0.1
        self.MAX_OBJECT_LIMIT = 12
        self.BUILD = "ALPHA"
        self.SIMILARITY_THRESHOLD = 0.8
        self.DEVICE = "0"
        self.VIDEO_SAMPLING_RATE = 1

    def build_response(self, message):
        return {"status": 200, "message": message}

    def __init__(self) -> None:
        self.__meta__()
        self.__topic_modeller_meta__()
        self.__scene_classification_meta__()
        logging.info(f"HAZEL'S RONIN {self.VERSION} {self.BUILD}")

    def predict_category(self, text):
        pre_trained_model_name = f"cardiffnlp/tweet-topic-21-multi"
        self.category_model = AutoModelForSequenceClassification.from_pretrained(pre_trained_model_name)
        pre_trained_tokenizer = AutoTokenizer.from_pretrained(pre_trained_model_name,  allow_long_sentences=True,)
        pre_trained_tokenizer.model_max_length = 512

        tokens = pre_trained_tokenizer(text, return_tensors='pt', truncation=True, max_length=512)

        if 'input_ids' not in tokens or len(tokens['input_ids']) == 0:
            raise ValueError("Tokenizer did not produce valid input tokens.")
        output = self.category_model(**tokens)
        return output
    
    def get_topic_name(self,text):
        if self.topic_model is not None:
            topics, _ = self.topic_model.transform(text)
            cluster_id = topics[0]
            cluster_name = list(self.topic_model.get_topic_info(cluster_id)['Name'])
            return (cluster_id,cluster_name)
        else:
            logging.error("> Model is not loaded. Make sure it is loaded or not.")

    def clean_output_pre_trained(self,output):
        try:
            class_mapping = self.category_model.config.id2label
            scores = expit(output[0][0].detach().numpy())
            topic_category_id, topic_category_id_score = sorted({category:score for score,category in zip(scores, class_mapping) 
                        if score > 0.5}.items(), key=lambda x:x[1])[-1]
            
            return topic_category_id
        except IndexError:
            topic_category_id = 19
            topic_category_id_score = 100
            return topic_category_id

    def use_pretrained_hugging_face_categorizer(self,text):
        unclean_output =self.predict_category(text)
        topic_category_id = self.clean_output_pre_trained(unclean_output)
        return topic_category_id
    
    def pre_process_text(self,documents):
        documents = [documents]
        clean_documents = []
        lemmatizer = WordNetLemmatizer()
        stop_words = set(stopwords.words('english'))

        for doc in documents:
            clean_sentences = ""
            sentences = sent_tokenize(doc)
            for sentence in sentences:
                clean_words = [lemmatizer.lemmatize(word.lower()) for word in word_tokenize(sentence) if
                               word not in stop_words]
                cln_stn = " ".join(clean_words)
                clean_sentences += (cln_stn.strip())

            clean_documents.append(clean_sentences)

        return clean_documents[0]
    
    def load_topic_model(self):

        model_path = f'topic_models/{self.topic_model_type}'
        if os.path.exists(model_path):
            self.topic_model = BERTopic.load(model_path,embedding_model= self.embedding_model)
            return 100
        else:
            logging.error(f"> Model not found in the default path. Check whether saved model exists at {model_path}")
            return 200
    # def fetch_video_object(self, video_id) -> None:
    #     try:
    #         return VideoEngineVideopost.objects.get(video_id=video_id)
    #     except VideoEngineVideopost.DoesNotExist:
    #         return None     

    # def fetch_video_path(self, video: VideoEngineVideopost) -> str:
    #     file = "".join(video.video_file.split("-"))
    #     media_path = os.path.join(settings.MEDIA_ROOT, file)
    #     return media_path
    
    # def get_video_status(self, video_obj:VideoEngineVideopost ) -> bool:
    #     return video_obj.gone_through_ml
    
    def calculate_time_duration(self, frame_index, fps):
        return frame_index / fps
    
    def get_video_fps(self, input_file):
        cap = cv2.VideoCapture(input_file)
        fps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        return fps

    def calculate_frame_similarity(self, frame1, frame2):
        gray_frame1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
        gray_frame2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
        
        percentage, _ = ssim(gray_frame1, gray_frame2, full=True)
        return percentage
    

    def make_scenes(self, input_file):
        cap = cv2.VideoCapture(input_file)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        scenes = {}
        scene_id = -1
        for i in range(0, frame_count - 1, int(fps * self.VIDEO_SAMPLING_RATE)):
            print(f"FRAME {i} - GENERATING SIMILARITY FOR SCENES..")
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret1, frame1 = cap.read()
            cap.set(cv2.CAP_PROP_POS_FRAMES, i + int(fps * self.VIDEO_SAMPLING_RATE))
            ret2, frame2 = cap.read()
            if not ret1 or not ret2:
                break

            similarity_index = self.calculate_frame_similarity(frame1, frame2)
   

            if similarity_index < self.SIMILARITY_THRESHOLD:
                scene_id = len(scenes)
                scenes[scene_id] = {
                    "frames": [i],
                    "similarity": similarity_index
                }
            elif scene_id != -1:
                scenes[scene_id]["frames"].append(i)
                scenes[scene_id]["similarity"] = max(
                    scenes[scene_id]["similarity"], similarity_index
                )

        cap.release()
        return scenes
    
    def get_scenes(self, scenes, fps):
        output = []
        for scene_id, scene_info in scenes.items():
            frame_indices = scene_info['frames']
            median_frame_index = int(median(frame_indices))
            time_occurance_seconds = self.calculate_time_duration(median_frame_index, fps)
            output.append((median_frame_index, time_occurance_seconds))
            
        return output
    
    def scenes_to_text(self,input_file ,scene_frames):
        cap = cv2.VideoCapture(input_file)
        images = []
        i = 0
        for frame_index in scene_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
            ret, frame = cap.read()

            if not ret:
                print(f"Failed to read frame {frame_index}")
                continue

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            i_image = Image.fromarray(frame_rgb)
            print(i)
            i += 1
            images.append(i_image)

        pixel_values = self.feature_extractor(images=images, return_tensors="pt").pixel_values
        pixel_values = pixel_values.to(self.device)

        output_ids = self.model.generate(pixel_values, **self.gen_kwargs)

        preds = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True, verbose = True)
        preds = [pred.strip() for pred in preds]
        return preds
    
    def _embed_scenes_to_transcript(self, transcript, scenes):
        embedded_transcript = transcript.copy()

        for chunk in embedded_transcript["chunks"]:
            for scene_start, scene_description in scenes:
                if chunk["timestamp"][0] <= scene_start <= chunk["timestamp"][1]:
                    text = chunk['text']
                    if "(on-screen:" in text:
                        splits = text.split(")")
                        scene = splits[0] + f",{scene_description})"
                        chunk['text'] = scene + splits[1]
                    else:
                        chunk["text"] = f"(on-screen: {scene_description}){chunk['text']}"

        return embedded_transcript

    
    def attach_descriptions_to_scenes(self, scenes_median_frames, descriptions):
        if len(scenes_median_frames) != len(descriptions):
            raise Exception("number of scenes doesn't match number of descriptions")
        
        formatted_scenes = []
        for scene, description in zip(scenes_median_frames, descriptions):
            new_scene = (scene[1], description)
            formatted_scenes.append(new_scene)
        
        return formatted_scenes
            
    
    def embed_scenes(self, median_frames, descriptions, path = "output.json"):
        if not os.path.exists(path):
            raise Exception(f"Transcribing or Translating output is not availabe at the path {path}.'")
        
        formatted_scenes = self.attach_descriptions_to_scenes(median_frames, descriptions)
        transcript = self._load_output_json(path)
        return self._embed_scenes_to_transcript(transcript, formatted_scenes)
    
   
    
    def _check_file_exists(self, file_path):
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File doesn't exist: {file_path}")

    def _run_subprocess(self, command):
        subprocess.run(command)
        
    def _load_output_json(self, path = "output.json"):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Output file not found: {self.OUTPUT_JSON}")

        with open(path) as json_file:
            return json.load(json_file)
        
    def _process_audio(self, input_file, task):
        self._check_file_exists(input_file)
        output_file = f"temp/{task.lower()}.wav"
        command = [
            'ffmpeg', '-i', input_file,
            '-ar', '16000',
            '-ac', '1',
            '-c:a', 'pcm_s16le', output_file,
            '-y'
        ]
        self._run_subprocess(command)
        return output_file if os.path.isfile(output_file) else None

    def extract_audio(self, input_file):
        return self._process_audio(input_file, 'audio')
    
    def resize_video(self, input_file):
        try:
            output_file = "temp/resized.mp4"
        
            if not os.path.exists(input_file):
                raise Exception("File doesn't exists")
            
            ffmpeg_resize = [
                'ffmpeg', '-i', input_file,
                '-s', 'hd480',
                '-c', 'copy', '-movflags', '+faststart', '-y', output_file     
            ]
            
            
            subprocess.run(ffmpeg_resize)

            if os.path.isfile(output_file):
                return output_file
            else:
                return None
        except Exception as e:
            raise e

    def _transcribe_or_translate(self, input_file, task):
        audio_path = self.extract_audio(input_file)
        command = [
            'insanely-fast-whisper',
            '--file-name', audio_path,
            #'--model-name', 'large-v3',
            #'--device-id', self.DEVICE,
            '--task', task
        ]
        self._run_subprocess(command)
        #return self._load_output_json()

    def transcribe_video(self, input_file):
        return self._transcribe_or_translate(input_file, 'transcribe')

    def translate_video(self, input_file):
        return self._transcribe_or_translate(input_file, 'translate')
    
    def _subtitles_format_seconds(self, seconds):
        whole_seconds = int(seconds)
        milliseconds = int((seconds - whole_seconds) * 1000)

        hours = whole_seconds // 3600
        minutes = (whole_seconds % 3600) // 60
        seconds = whole_seconds % 60

        return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"

    def _subtitles_format_chunk(self, chunk, index):
        text = chunk['text']
        start, end = chunk['timestamp'][0], chunk['timestamp'][1]
        start_format, end_format = self.format_seconds(start), self.format_seconds(end)
        return f"{index}\n{start_format} --> {end_format}\n{text}\n\n"

    def make_subtitles(self, input_file = "output.json"):
        try:
            output_file = "temp/resized.mp4"
        
            self._check_file_exists(input_file)
            with open(input_file, 'r') as file:
                data = json.load(file)
                
            string = ""
            for index, chunk in enumerate(data['chunks'], 1):
                entry = self._subtitles_format_chunk(chunk, index)
                string += entry

            with open(output_file, 'w', encoding='utf-8') as file:
                file.write(string)
            
        except Exception as e:
            raise e
    
    def recognize_audio_langauge(self, input_file):
        try:
            self._check_file_exists(input_file)
            model = whisper.load_model("base")
            audio = whisper.load_audio(input_file)
            audio = whisper.pad_or_trim(audio)
            mel = whisper.log_mel_spectrogram(audio).to(model.device)
            _, probs = model.detect_language(mel)
            return max(probs, key=probs.get)
        except Exception as e:
            raise e


    def update_video_post_data(self):
        pass

    def upload_subtitles(self):
        pass        

    def fine_tune_llm_for_videos(self):
        pass
    
    def save_transcription(self):
        pass
    
    def save_translation(self):
        pass

    def save_modified_llm(self):
        pass

     
ronin = Ronin()


ImportError: libcudnn.so.8: cannot open shared object file: No such file or directory

In [2]:
ronin.extract_audio("test.mp4")
ronin.resize_video("test.mp4")

NameError: name 'ronin' is not defined

In [ ]:
ronin.recognize_audio_langauge("temp/audio.wav")

In [ ]:
output = ronin.transcribe_video("test.mp4")

In [ ]:
f = open("output.json")
output = json.load(f)
output

In [ ]:
scenes = ronin.make_scenes("temp/resized.mp4")
fps = ronin.get_video_fps("temp/resized.mp4")


In [ ]:
scenes_median_frames = ronin.get_scenes(scenes,fps)

In [ ]:
scene_descriptions = ronin.scenes_to_text("temp/resized.mp4",[i for i, y in scenes_median_frames])

In [ ]:
scenes

In [3]:
output

NameError: name 'output' is not defined

In [4]:
em = ronin.embed_scenes(scenes_median_frames, scene_descriptions)

NameError: name 'ronin' is not defined

In [ ]:
def make_transcript_text(output):
    text = ""
    for scene in output['chunks']:
        text += scene['text']
    return text

tt  = make_transcript_text(em)
tt

In [ ]:


def start_topic_modelling(text_content):
    ronin.load_topic_model()
    clean_document = ronin.pre_process_text(text_content)
    topic_cluster_id = ronin.get_topic_name(clean_document)
    topic_category_id = ronin.use_pretrained_hugging_face_categorizer(clean_document)
    return (topic_cluster_id[0],topic_cluster_id[1], topic_category_id)


start_topic_modelling(tt)

In [ ]:
conda install -c conda-forge cudatoolkit=11.2 cudnn=8.1

In [ ]:
ronin.topic_model.visualize_topics()

In [5]:
!pip install nbformat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.1 MB/s eta 0:00:00


In [6]:
TOPICS = {
    "Cars and vehicles": 1,
    "Comedy": 2,
    "Education": 3,#
    "Entertainment": 4,#
    "Film and animation": 5,
    "Gaming": 6,#
    "How-to and style": 7,
    "Music": 8,#
    "News and politics": 9,#
    "Non-profits and activism": 10,
    "People and blogs": 11,
    "Pets and animals": 12,
    "Science and technology": 13,#
    "Sport": 14,#
    "Travel and events": 15#
}

In [7]:
ronin.load_topic_model()

NameError: name 'ronin' is not defined

In [10]:
topics = ronin.topic_model.topic_labels_

NameError: name 'ronin' is not defined

In [ ]:
topic_keys = list(topics.values())

s = []
for i in topic_keys:
    x = i.split("_")[1:]
    s += x

In [ ]:
s = list(set(s))

In [9]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired


zeroshot_topic_list = ["News", "Video Games", "Music", "Entertainment", "Education","Science and Technology" ,"Movies","People","Pets and Animals","Vehicles" ,"Sports","Travel"]

topic_model = BERTopic(
    embedding_model=SentenceTransformer("all-mpnet-base-v2"), 
    zeroshot_topic_list=zeroshot_topic_list,
    zeroshot_min_similarity=0,
    nr_topics=len(zeroshot_topic_list),
    representation_model=KeyBERTInspired()
)
topics, _ = topic_model.fit_transform(s)

/home/user/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: libcudnn.so.8: cannot open shared object file: No such file or directory

In [ ]:
topic_model.get_topic_info()

In [ ]:
topics_to_merge = [[12, 14, 15],
[11,16]
]

topic_model.merge_topics(s, topics_to_merge)


In [ ]:
topic_model.visualize_topics()

NameError: name 'topic_model' is not defined

In [ ]:
ss = BERTopic(
    embedding_model=SentenceTransformer("all-distilroberta-v1"), 
    zeroshot_topic_list=zeroshot_topic_list,
    representation_model=KeyBERTInspired()
)
topics, _ = ss.fit_transform(w)

In [ ]:
ss.get_topic_info()

In [ ]:
TOPIC_MAP = {
    "News": [12],
    "Gaming": [9],
    "Music": [11],
    "Entertainment": [0, 6, 14],
    "Education": [10],
    "Science and Technology": [15],
    "Movies": [6],
    "People": [2, 4, 13],
    "Pets and Animals": [13],
    "Vehicles": [],  # This category wasn't present in the original mapping, so it's empty
    "Sports": [10, 16],
    "Travel": [17]
}


In [ ]:
list(TOPIC_MAP.keys())